# 서울 아파트 전월세 데이터 EDA & Feature Engineering

## 1. 분석 목적

서울 아파트 매매가격 예측 모델에 **임대시장 상황을 반영하기 위해** 아파트 전월세 실거래 데이터를 수집하고 전처리한다.

단순히 현재 전세가격을 사용하는 것이 아니라, 실제 매매 거래 시점 이전에 확인할 수 있었던 **과거 임대시장 정보**를 생성하는 것이 목적이다.

최종적으로 다음과 같은 피처를 매매 데이터에 결합한다.

- 전월 전세가격 수준
- 과거 3개월·6개월 전세가격 수준
- 전월 전세 거래량
- 과거 3개월·6개월 전세 거래량
- 전세가율
- 단지 및 법정동 단위 임대시장 흐름

---

In [29]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data" / "v2_5year_seoul"
RAW_RENT_DIR = DATA_DIR / "raw" / "rent"
PROCESSED_DIR = DATA_DIR / "processed"

RENT_RAW_PATH = PROCESSED_DIR / "apt_rent_raw.parquet"
TRADE_FEATURE_PATH = PROCESSED_DIR / "df_fe.parquet"

RENT_GU_PATH = PROCESSED_DIR / "rent_gu_monthly.parquet"
RENT_DONG_PATH = PROCESSED_DIR / "rent_dong_monthly.parquet"
RENT_APT_PATH = PROCESSED_DIR / "rent_apt_monthly.parquet"
TRADE_RENT_FEATURE_PATH = PROCESSED_DIR / "df_fe_with_rent.parquet"

MODEL_START = pd.Timestamp("2021-07-01")
MODEL_END = pd.Timestamp("2026-06-01")
ROLLING_WINDOWS = (3, 6)
PRICE_MIN_MONTHS = {3: 2, 6: 3}
APT_MIN_TRADES = 3

# 마지막 저장 셀에서만 사용한다. 검토 중에는 False로 둔다.
SAVE_OUTPUTS = False

analysis_config = pd.Series(
    {
        "모델 시작월": MODEL_START,
        "모델 종료월": MODEL_END,
        "rolling 기간": ROLLING_WINDOWS,
        "단지 최소 거래량": APT_MIN_TRADES,
        "결과 저장": SAVE_OUTPUTS,
    },
    name="설정",
)
display(analysis_config.to_frame())

,설정
모델 시작월,2021-07-01 00:00:00
모델 종료월,2026-06-01 00:00:00
rolling 기간,"(3, 6)"
단지 최소 거래량,3
결과 저장,False


## 2. 데이터 수집

### 2.1 데이터 출처

국토교통부 아파트 전월세 실거래가 API를 사용한다.

수집 범위는 다음과 같다.

- 지역: 서울특별시 25개 자치구
- 기간: 2021-01 ~ 2026-06
- 총 기간: 66개월
- 수집 단위: 자치구 × 월
- 총 요청 단위: 25개 자치구 × 66개월 = 1,650개

각 자치구·월별 응답은 개별 CSV checkpoint로 저장하고, 전체 데이터를 하나의 parquet 파일로 통합한다.

### 2.2 수집 기간을 2021-01부터 시작한 이유

실제 매매가격 모델링 기간은 다음과 같다.

> 2021-07 ~ 2026-06

하지만 3개월·6개월 과거 시장정보를 생성하려면 모델링 시작 시점보다 앞선 데이터가 필요하다.

- 2021-01 ~ 2021-06: lag·rolling 피처 계산을 위한 준비 기간
- 2021-07 ~ 2026-06: 실제 모델링 기간

따라서 모델링 시작일보다 6개월 앞선 2021-01부터 전월세 데이터를 수집한다.

In [30]:
required_paths = [RENT_RAW_PATH, TRADE_FEATURE_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"필수 입력 파일이 없습니다: {missing_paths}")

rent_raw = pd.read_parquet(RENT_RAW_PATH)

checkpoint_records = []
for path in sorted(RAW_RENT_DIR.glob("rent_*.csv")):
    parts = path.stem.split("_")
    if len(parts) == 3 and len(parts[1]) == 5 and len(parts[2]) == 6:
        checkpoint_records.append(
            {
                "시군구코드": parts[1],
                "거래연월": parts[2],
                "파일": path.name,
            }
        )

checkpoint_df = pd.DataFrame(checkpoint_records)
expected_months = pd.period_range("2021-01", "2026-06", freq="M")
checkpoint_districts = (
    sorted(checkpoint_df["시군구코드"].unique())
    if not checkpoint_df.empty else []
)
expected_requests = len(checkpoint_districts) * len(expected_months)

collection_summary = pd.Series(
    {
        "checkpoint 파일 수": len(checkpoint_df),
        "확인된 자치구 수": len(checkpoint_districts),
        "기대 월 수": len(expected_months),
        "기대 요청 수": expected_requests,
        "통합 parquet 행 수": len(rent_raw),
        "통합 parquet 컬럼 수": len(rent_raw.columns),
    },
    name="값",
)

display(collection_summary.to_frame())
display(rent_raw.head())

,값
checkpoint 파일 수,1650
확인된 자치구 수,25
기대 월 수,66
기대 요청 수,1650
통합 parquet 행 수,1415810
통합 parquet 컬럼 수,28


,aptNm,aptSeq,buildYear,contractTerm,contractType,dealDay,dealMonth,dealYear,deposit,excluUseAr,...,roadnmbubun,roadnmcd,roadnmseq,roadnmsggcd,sggCd,umdNm,useRRRight,수집_자치구,수집_시군구코드,수집_거래연월
0,경희궁롯데캐슬,11110-2519,2019,None,None,30,1,2021,"56,000",59.6737,...,00000,3000008,6,11110,11110,무악동,None,종로구,11110,202101
1,아남1,11110-25,1995,None,None,22,1,2021,"53,550",84.8,...,00000,3005008,7,11110,11110,명륜2가,None,종로구,11110,202101
2,두산,11110-34,1999,None,None,30,1,2021,"36,330",59.95,...,00000,4100390,1,11110,11110,창신동,None,종로구,11110,202101
3,광화문스페이스본(101동~105동),11110-2203,2008,None,None,9,1,2021,"80,000",70.8,...,00000,4100135,3,11110,11110,사직동,None,종로구,11110,202101
4,창신쌍용1,11110-37,1992,None,None,23,1,2021,"38,800",54.7,...,00000,4100065,1,11110,11110,창신동,None,종로구,11110,202101


## 3. Raw Data 검증

전체 데이터를 통합한 뒤 다음 항목을 먼저 확인한다.

- 전체 행 수와 컬럼 수
- 데이터 기간과 월 개수
- 서울 25개 자치구 포함 여부
- 컬럼별 결측치와 자료형
- 완전히 동일한 행의 중복
- 주요 변수의 이상값

핵심 분석 컬럼은 다음과 같다.

| 컬럼 | 의미 |
|---|---|
| `aptSeq` | 아파트 단지 식별자 |
| `aptNm` | 아파트명 |
| `sggCd` | 시군구 코드 |
| `umdNm` | 법정동 |
| `dealYear` | 계약연도 |
| `dealMonth` | 계약월 |
| `dealDay` | 계약일 |
| `deposit` | 보증금 |
| `monthlyRent` | 월세 |
| `excluUseAr` | 전용면적 |
| `floor` | 층 |
| `buildYear` | 건축연도 |

프로파일·가격·면적·날짜에 필요한 핵심 변수를 우선 검증한 뒤 시장 피처를 생성한다.

---

In [31]:
required_raw_cols = [
    "aptSeq",
    "aptNm",
    "sggCd",
    "umdNm",
    "dealYear",
    "dealMonth",
    "dealDay",
    "deposit",
    "monthlyRent",
    "excluUseAr",
    "floor",
    "buildYear",
    "수집_자치구",
    "수집_시군구코드",
]
missing_required_cols = [
    col for col in required_raw_cols if col not in rent_raw.columns
]
if missing_required_cols:
    raise KeyError(f"필수 컬럼이 없습니다: {missing_required_cols}")

raw_deal_month = pd.to_datetime(
    {
        "year": pd.to_numeric(rent_raw["dealYear"], errors="coerce"),
        "month": pd.to_numeric(rent_raw["dealMonth"], errors="coerce"),
        "day": 1,
    },
    errors="coerce",
)

schema_check = pd.DataFrame(
    {
        "dtype": rent_raw.dtypes.astype(str),
        "missing": rent_raw.isna().sum(),
        "missing_ratio": rent_raw.isna().mean(),
    }
).sort_values("missing_ratio", ascending=False)

raw_quality_summary = pd.Series(
    {
        "행 수": len(rent_raw),
        "컬럼 수": len(rent_raw.columns),
        "최초 거래월": raw_deal_month.min(),
        "최종 거래월": raw_deal_month.max(),
        "거래월 수": raw_deal_month.dt.to_period("M").nunique(),
        "자치구 수": rent_raw["수집_시군구코드"].nunique(),
        "완전 동일 행": rent_raw.duplicated().sum(),
    },
    name="값",
)

display(raw_quality_summary.to_frame())
display(schema_check)

,값
행 수,1415810
컬럼 수,28
최초 거래월,2021-01-01 00:00:00
최종 거래월,2026-06-01 00:00:00
거래월 수,66
자치구 수,25
완전 동일 행,60926


,dtype,missing,missing_ratio
preMonthlyRent,object,1277220,0.902113
useRRRight,object,1219170,0.861111
preDeposit,object,993844,0.701961
contractTerm,object,263415,0.186053
contractType,object,252760,0.178527
roadnmbcd,object,191569,0.135307
roadnmbonbun,object,1491,0.001053
roadnmbubun,object,1491,0.001053
roadnm,object,1491,0.001053
roadnmcd,object,1491,0.001053


## 4. 결측치 처리 방향

결측치가 많은 컬럼은 주로 계약 갱신 관련 정보다.

- `preDeposit`
- `preMonthlyRent`
- `useRRRight`
- `contractTerm`
- `contractType`

이 변수들은 초기 전세가격·거래량 피처 생성에 반드시 필요한 값이 아니다. 따라서 근거 없이 임의 대체하지 않고 원본 결측 상태를 유지한다.

반면 `aptSeq`, `sggCd`, `umdNm`, `aptNm`처럼 프로파일 연결에 필요한 지역·단지 식별 변수는 별도로 완전성을 확인한다.

---

In [32]:
renewal_cols = [
    "preDeposit",
    "preMonthlyRent",
    "useRRRight",
    "contractTerm",
    "contractType",
]
profile_cols = ["aptSeq", "sggCd", "umdNm", "aptNm"]

missing_policy = pd.DataFrame(
    {
        "결측건수": rent_raw[renewal_cols + profile_cols].isna().sum(),
        "결측비율": rent_raw[renewal_cols + profile_cols].isna().mean(),
        "처리방향": [
            "원본 결측 유지" if col in renewal_cols else "결합 전 완전성 확인"
            for col in renewal_cols + profile_cols
        ],
    }
).sort_values("결측비율", ascending=False)

critical_profile_missing = rent_raw[profile_cols].isna().sum()
if critical_profile_missing.sum() > 0:
    print("주의: 지역·단지 식별 컬럼에 결측치가 있습니다.")

display(missing_policy)

,결측건수,결측비율,처리방향
preMonthlyRent,1277220,0.902113,원본 결측 유지
useRRRight,1219170,0.861111,원본 결측 유지
preDeposit,993844,0.701961,원본 결측 유지
contractTerm,263415,0.186053,원본 결측 유지
contractType,252760,0.178527,원본 결측 유지
aptSeq,0,0.000000,결합 전 완전성 확인
sggCd,0,0.000000,결합 전 완전성 확인
umdNm,0,0.000000,결합 전 완전성 확인
aptNm,0,0.000000,결합 전 완전성 확인


## 5. 중복 데이터 검토

완전히 동일해 보이는 거래가 일부 존재할 수 있다. 그러나 전월세 실거래 공개자료에는 계약을 유일하게 구분하는 행 번호가 없으므로, 같은 조건의 두 행이 실제 중복인지 서로 다른 동·호수 계약인지 단정할 수 없다.

다음 거래조건을 기준으로 중복 구조를 확인한다.

- `aptSeq`
- `dealYear`, `dealMonth`, `dealDay`
- `excluUseAr`
- `floor`
- `deposit`
- `monthlyRent`

### 중복 처리 원칙

> 명확한 오류라고 판단할 근거가 없는 거래는 유지한다.

이후 거래량 피처를 생성하므로 실제 거래를 잘못 제거하는 쪽이 더 큰 문제를 만들 수 있다. 중복 표본과 갱신 관련 컬럼을 함께 확인한 뒤 별도의 제거 근거가 생길 때만 처리한다.

In [33]:
duplicate_key = [
    "aptSeq",
    "dealYear",
    "dealMonth",
    "dealDay",
    "excluUseAr",
    "floor",
    "deposit",
    "monthlyRent",
]

exact_dup_mask = rent_raw.duplicated(keep=False)
condition_dup_mask = rent_raw.duplicated(
    subset=duplicate_key,
    keep=False,
)
condition_dup_groups = (
    rent_raw.loc[condition_dup_mask]
    .groupby(duplicate_key, dropna=False)
    .ngroups
)

duplicate_summary = pd.Series(
    {
        "완전 동일 중복 포함 행": exact_dup_mask.sum(),
        "거래조건 중복 포함 행": condition_dup_mask.sum(),
        "거래조건 중복 그룹": condition_dup_groups,
        "자동 삭제 행": 0,
    },
    name="건수",
)

duplicate_detail_cols = duplicate_key + [
    "contractType",
    "contractTerm",
    "useRRRight",
    "preDeposit",
    "preMonthlyRent",
]

display(duplicate_summary.to_frame())
display(
    rent_raw.loc[condition_dup_mask, duplicate_detail_cols]
    .sort_values(duplicate_key)
    .head(20)
)

,건수
완전 동일 중복 포함 행,116143
거래조건 중복 포함 행,243507
거래조건 중복 그룹,115911
자동 삭제 행,0


,aptSeq,dealYear,dealMonth,dealDay,excluUseAr,floor,deposit,monthlyRent,contractType,contractTerm,useRRRight,preDeposit,preMonthlyRent
3023,11110-101,2022,7,12,187.9,2,"110,000",0,None,None,None,None,None
3149,11110-101,2022,7,12,187.9,2,"110,000",0,신규,22.09~24.09,None,None,None
1547,11110-102,2021,10,1,48.54,5,"1,000",100,신규,21.10~23.10,None,None,None
1584,11110-102,2021,10,1,48.54,5,"1,000",100,신규,21.10~23.10,None,None,None
1787,11110-102,2021,12,28,48.54,3,"30,000",0,None,None,None,None,None
1832,11110-102,2021,12,28,48.54,3,"30,000",0,신규,22.02~24.02,None,None,None
7579,11110-102,2024,8,2,45.5,9,"26,000",30,신규,24.09~26.09,None,None,None
7610,11110-102,2024,8,2,45.5,9,"26,000",30,신규,24.09~26.09,None,None,None
9067,11110-102,2025,4,1,45.5,2,"30,000",20,신규,25.05~27.05,None,None,None
9168,11110-102,2025,4,1,45.5,2,"30,000",20,신규,25.05~27.05,None,None,None


## 6. 기본 전처리

### 6.1 숫자형 변환

다음 컬럼을 숫자형으로 변환한다.

- 계약연도·계약월·계약일
- 보증금·월세
- 전용면적·층·건축연도

API에서 문자열로 수집된 숫자를 정리하고 금액에 포함된 쉼표를 제거한다. 변환할 수 없는 값은 결측값으로 두어 이후 품질 점검에서 확인한다.

### 6.2 날짜 변수 생성

계약연도·계약월·계약일을 이용해 실제 `거래일자`를 생성한다. 월별 시장 피처를 만들기 위해 각 거래일을 해당 월의 첫날로 통일한 `거래월일자`도 생성한다.

예시:

> 2024-03-17 → 2024-03-01

모든 전월세·매매 월별 집계는 이 월 기준일을 공통 키로 사용한다.

---

In [34]:
rent = rent_raw.copy()
rent = rent.replace(["nan", "None", "null", ""], pd.NA)

numeric_cols = [
    "dealYear",
    "dealMonth",
    "dealDay",
    "deposit",
    "monthlyRent",
    "excluUseAr",
    "floor",
    "buildYear",
]
for col in numeric_cols:
    rent[col] = pd.to_numeric(
        rent[col].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )

text_cols = [
    "aptNm",
    "umdNm",
    "contractType",
    "contractTerm",
    "useRRRight",
    "수집_자치구",
]
for col in text_cols:
    rent[col] = rent[col].astype("string").str.strip()

rent["sggCd"] = rent["sggCd"].astype("string").str.split(".").str[0].str.zfill(5)
rent["수집_시군구코드"] = (
    rent["수집_시군구코드"]
    .astype("string")
    .str.split(".")
    .str[0]
    .str.zfill(5)
)
rent["aptSeq"] = rent["aptSeq"].astype("string").str.strip()

rent["거래일자"] = pd.to_datetime(
    {
        "year": rent["dealYear"],
        "month": rent["dealMonth"],
        "day": rent["dealDay"],
    },
    errors="coerce",
)
rent["거래월일자"] = rent["거래일자"].dt.to_period("M").dt.to_timestamp()
rent["연식"] = rent["dealYear"] - rent["buildYear"]

invalid_age = rent["연식"].lt(0) | rent["연식"].gt(100)
rent.loc[invalid_age, "연식"] = np.nan

preprocess_summary = pd.Series(
    {
        "행 수": len(rent),
        "거래일자 결측": rent["거래일자"].isna().sum(),
        "비정상 연식": invalid_age.sum(),
        "시군구코드 불일치": (
            rent["sggCd"] != rent["수집_시군구코드"]
        ).sum(),
    },
    name="건수",
)
display(preprocess_summary.to_frame())

,건수
행 수,1415810
거래일자 결측,0
비정상 연식,9171
시군구코드 불일치,0


## 7. 전세·월세 분리

월세 금액을 기준으로 계약 유형을 구분한다.

- `monthlyRent == 0` → 전세
- `monthlyRent > 0` → 월세

초기 매매가격 예측 피처에서는 매매가격과 관계가 더 직접적인 전세시장을 우선 사용한다. 월세 데이터는 제거하지 않고 보존하며, 이후 보증부월세 환산 또는 추가 시장 피처에 활용할 수 있다.

---

In [35]:
rent["계약유형"] = pd.Series(pd.NA, index=rent.index, dtype="string")
rent.loc[rent["monthlyRent"].eq(0), "계약유형"] = "전세"
rent.loc[rent["monthlyRent"].gt(0), "계약유형"] = "월세"

contract_summary = (
    rent["계약유형"]
    .value_counts(dropna=False)
    .rename_axis("계약유형")
    .to_frame("거래량")
)

monthly_contract = (
    rent.dropna(subset=["거래월일자", "계약유형"])
    .groupby(["거래월일자", "계약유형"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
monthly_contract["전세비율"] = (
    monthly_contract.get("전세", 0) / monthly_contract.sum(axis=1)
)

display(contract_summary)
display(monthly_contract.tail(12))

,거래량
계약유형,
전세,805108
월세,610702


계약유형,월세,전세,전세비율
거래월일자,,,
2025-07-01,9718,12006,0.552661
2025-08-01,8937,11328,0.558993
2025-09-01,9425,12058,0.561281
2025-10-01,9141,11084,0.548035
2025-11-01,9797,12156,0.553728
2025-12-01,12590,12557,0.499344
2026-01-01,11461,12382,0.519314
2026-02-01,9161,9951,0.520668
2026-03-01,10993,10823,0.496104


## 8. 전세가격 피처 생성

단순 보증금은 주택 면적 차이의 영향을 크게 받으므로 지역·단지 간 비교에는 평당가를 사용한다.

> 전세평당가 = 보증금 ÷ (전용면적 ÷ 3.3058)

전용면적이 0 이하이거나 보증금이 0인 값은 계산 전후에 별도로 확인한다.

In [36]:
valid_jeonse = (
    rent["계약유형"].eq("전세")
    & rent["deposit"].gt(0)
    & rent["excluUseAr"].gt(0)
    & rent["거래월일자"].notna()
)

jeonse = rent.loc[valid_jeonse].copy()
jeonse["전세평당가"] = (
    jeonse["deposit"] / (jeonse["excluUseAr"] / 3.3058)
)

jeonse_price_summary = jeonse["전세평당가"].describe(
    percentiles=[0.001, 0.005, 0.01, 0.5, 0.99, 0.995, 0.999]
)
jeonse_filter_summary = pd.Series(
    {
        "전체 전세": rent["계약유형"].eq("전세").sum(),
        "분석 가능 전세": len(jeonse),
        "보증금 0 이하 제외": (
            rent["계약유형"].eq("전세") & rent["deposit"].le(0)
        ).sum(),
        "면적 0 이하 제외": (
            rent["계약유형"].eq("전세") & rent["excluUseAr"].le(0)
        ).sum(),
    },
    name="건수",
)

display(jeonse_filter_summary.to_frame())
display(jeonse_price_summary.to_frame("전세평당가"))

,건수
전체 전세,805108
분석 가능 전세,805108
보증금 0 이하 제외,0
면적 0 이하 제외,0


,전세평당가
count,805108.0
mean,2486.299536
std,1128.603072
min,0.332019
0.1%,294.958895
0.5%,506.959638
1%,625.966241
50%,2293.146504
99%,6329.74711
99.5%,6984.139593


## 9. 전월세 EDA

본격적인 시장 피처 생성 전에 전월세시장의 전체 구조와 데이터 품질을 확인한다.

### 9.1 계약 유형

- 전체 전월세 거래량
- 전세 거래량과 월세 거래량
- 전세·월세 비율 변화

### 9.2 시간 변화

- 연도별·월별 거래량
- 전세 거래량 추이
- 전세 평당가 추이

### 9.3 지역별 분석

- 자치구별 전세 거래량
- 자치구별 전세 평당가
- 법정동별 전세가격
- 지역별 전세가격 분포

### 9.4 주택 특성

- 전용면적별 전세가격
- 연식별 전세가격
- 층별 전세가격
- 면적과 전세보증금의 관계

EDA 결과는 분포를 이해하고 후속 시장 피처의 집계 단위와 fallback 구조를 결정하는 근거로 사용한다.

### 9.5 이상치 상세 점검

다음과 같은 오류·극단값 후보를 확인한다.

- 보증금이 0인 전세 계약
- 음수 또는 비정상적인 월세
- 0 이하이거나 비정상적으로 큰 전용면적
- 거래연도보다 늦은 건축연도
- 0층 이하의 층 정보
- 극단적으로 낮거나 높은 전세평당가

명확한 오류 기준이 없는 값은 자동 삭제하지 않는다. 기술통계, 분위수와 원본 표본을 함께 확인하고 실제 제거가 필요한 이상치만 판단한다.

In [37]:
quality_check = pd.Series(
    {
        "보증금_0": rent["deposit"].eq(0).sum(),
        "월세_음수": rent["monthlyRent"].lt(0).sum(),
        "면적_0이하": rent["excluUseAr"].le(0).sum(),
        "층_0이하": rent["floor"].le(0).sum(),
        "건축년도_거래년도초과": (
            rent["buildYear"] > rent["dealYear"]
        ).sum(),
    },
    name="건수",
)

eda_monthly = (
    rent.groupby("거래월일자")
    .agg(
        전체거래량=("계약유형", "size"),
        전세거래량=("계약유형", lambda s: s.eq("전세").sum()),
        월세거래량=("계약유형", lambda s: s.eq("월세").sum()),
    )
    .join(
        jeonse.groupby("거래월일자")["전세평당가"]
        .median()
        .rename("전세중위평당가")
    )
    .sort_index()
)

eda_gu = (
    jeonse.groupby(["sggCd", "수집_자치구"])
    .agg(
        전세거래량=("전세평당가", "size"),
        전세중위평당가=("전세평당가", "median"),
    )
    .sort_values("전세중위평당가", ascending=False)
)

extreme_cols = [
    "aptNm",
    "수집_자치구",
    "umdNm",
    "거래일자",
    "deposit",
    "excluUseAr",
    "floor",
    "전세평당가",
]
extreme_jeonse = pd.concat(
    {
        "하위": jeonse.nsmallest(10, "전세평당가")[extreme_cols],
        "상위": jeonse.nlargest(10, "전세평당가")[extreme_cols],
    },
    names=["구간", "원본인덱스"],
)

display(quality_check.to_frame())
display(eda_monthly.tail(12))
display(eda_gu)
display(extreme_jeonse)

,건수
보증금_0,2325
월세_음수,0
면적_0이하,0
층_0이하,173
건축년도_거래년도초과,9171


,전체거래량,전세거래량,월세거래량,전세중위평당가
거래월일자,,,,
2025-07-01,21724,12006,9718,2470.852281
2025-08-01,20265,11328,8937,2369.946649
2025-09-01,21483,12058,9425,2410.149037
2025-10-01,20225,11084,9141,2476.806855
2025-11-01,21953,12156,9797,2608.03414
2025-12-01,25147,12557,12590,2565.287118
2026-01-01,23843,12382,11461,2590.968372
2026-02-01,19112,9951,9161,2478.786639
2026-03-01,21816,10823,10993,2484.318637


,,전세거래량,전세중위평당가
sggCd,수집_자치구,,
11680,강남구,67485,3242.3932
11650,서초구,49316,3241.795725
11200,성동구,33652,2790.070347
11140,중구,10548,2757.590924
11170,용산구,18771,2726.272385
11215,광진구,15739,2704.235221
11710,송파구,68521,2681.931621
11440,마포구,35782,2671.856426
11110,종로구,5630,2535.063903


aptNm 수집_자치구  umdNm       거래일자  deposit  excluUseAr  \
구간 원본인덱스                                                                  
하위 95938    금호벽산(임대)(미공시)    성동구  금호동1가 2023-09-30        3       29.87   
   540988              충정   서대문구  충정로3가 2026-06-10     1000       86.61   
   290897    삼성래미안트리베라2단지    강북구    미아동 2021-01-30     1300       59.38   
   291642         미아1차래미안    강북구    미아동 2021-05-15     1500        59.9   
   290942    삼성래미안트리베라2단지    강북구    미아동 2021-01-13     1500       59.33   
   292391        효성네오인텔리안    강북구     번동 2021-08-31     2000       71.05   
   677383        우장산힐스테이트    강서구   내발산동 2021-04-08     2450       84.98   
   839901         래미안당산1차   영등포구    당산동 2021-04-17     1800       59.97   
   298295      성원그린(103동)    강북구    우이동 2023-05-24     2800       84.79   
   237671            면목임대    중랑구    면목동 2026-05-20      844        25.2   
상위 1195918          PH129    강남구    청담동 2026-02-13  1385000      273.96   
   145944             어울채    광진구    자양동 2025-04-26   180000       39.02   
   1059715          메이플자이    서초구    잠원동 2025-07-02   265000     59.5368   
   106001       아크로서울포레스트    성동구  성수동1가 2024-09-04   700000     159.603   
   1023847        래미안원베일리    서초구    반포동 2023-06-07   259000       59.96   
   1060146          메이플자이    서초구    잠원동 2025-07-02   252500     59.5368   
   1191544           청담르엘    강남구    청담동 2025-12-05   470000    111.9779   
   1026016        래미안원베일리    서초구    반포동 2023-07-27   780000      185.95   
   1063639        래미안원베일리    서초구    반포동 2025-10-30   780000      185.95   
   1064031        래미안원베일리    서초구    반포동 2025-10-30   780000      185.95   

            floor         전세평당가  
구간 원본인덱스                         
하위 95938       16      0.332019  
   540988       3     38.168803  
   290897      14     72.373526  
   291642       4     82.782972  
   290942       8     83.578291  
   292391       9     93.055595  
   677383       6     95.307249  
   839901       7     99.223612  
   298295       3    109.166647  
   237671      11    110.718063  
상위 1195918     18  16712.414221  
   145944      19  15249.718093  
   1059715     29  14714.210371  
   106001      42  14498.850272  
   1023847     15  14279.556371  
   1060146     29  14020.143844  
   1191544     22  13875.291464  
   1026016     34  13866.759882  
   1063639     34  13866.759882  
   1064031     34  13866.759882

## 10. 과거 전세시장 피처 생성

매매가격을 예측하는 시점의 전세가격을 그대로 사용하면 동일 시점 또는 미래 정보가 포함될 수 있다. 따라서 모든 시장 피처는 최소 1개월 이전 정보만 사용한다.

- 현재월 전세시장 집계: 사용하지 않음
- 전월 및 그 이전 시장 집계: 사용 가능

---

In [38]:
ALL_MONTHS = pd.date_range(
    rent["거래월일자"].min(),
    rent["거래월일자"].max(),
    freq="MS",
)


def complete_month_panel(monthly, group_cols, count_cols):
    groups = monthly[group_cols].drop_duplicates()
    months = pd.DataFrame({"거래월일자": ALL_MONTHS})
    grid = groups.merge(months, how="cross")
    panel = grid.merge(
        monthly,
        on=group_cols + ["거래월일자"],
        how="left",
        validate="one_to_one",
    )
    for col in count_cols:
        panel[col] = panel[col].fillna(0).astype("int64")
    return panel.sort_values(group_cols + ["거래월일자"]).reset_index(drop=True)


def add_lag_features(
    panel,
    group_cols,
    prefix,
    market_name,
    price_col,
    volume_col,
):
    result = panel.sort_values(group_cols + ["거래월일자"]).copy()
    grouped = result.groupby(group_cols, dropna=False, group_keys=False)

    result[f"{prefix}_전월_{market_name}평당가"] = (
        grouped[price_col].shift(1)
    )
    result[f"{prefix}_전월_{market_name}거래량"] = (
        grouped[volume_col].shift(1)
    )

    for window in ROLLING_WINDOWS:
        price_min_months = PRICE_MIN_MONTHS[window]
        result[f"{prefix}_과거{window}개월_{market_name}평당가"] = (
            grouped[price_col].transform(
                lambda s: s.shift(1).rolling(
                    window=window,
                    min_periods=price_min_months,
                ).median()
            )
        )
        result[f"{prefix}_과거{window}개월_{market_name}거래량"] = (
            grouped[volume_col].transform(
                lambda s: s.shift(1).rolling(
                    window=window,
                    min_periods=window,
                ).sum()
            )
        )

    return result


print("전체 월 범위:", ALL_MONTHS.min(), "~", ALL_MONTHS.max())
print("lag 원칙: 모든 피처에 shift(1) 적용")

전체 월 범위: 2021-01-01 00:00:00 ~ 2026-06-01 00:00:00
lag 원칙: 모든 피처에 shift(1) 적용


## 11. 법정동 단위 피처

먼저 충분한 거래량을 확보할 수 있는 법정동 단위 피처를 생성한다.

- `법정동_전월_전세평당가`
- `법정동_과거3개월_전세평당가`
- `법정동_과거6개월_전세평당가`
- `법정동_전월_전세거래량`
- `법정동_과거3개월_전세거래량`
- `법정동_과거6개월_전세거래량`

가격은 평균보다 이상치의 영향을 적게 받는 중앙값을 우선 사용한다.

---

In [39]:
gu_monthly = (
    jeonse.groupby(["sggCd", "거래월일자"])
    .agg(
        월중위전세평당가=("전세평당가", "median"),
        월전세거래량=("전세평당가", "size"),
    )
    .reset_index()
)
gu_panel = complete_month_panel(
    gu_monthly,
    group_cols=["sggCd"],
    count_cols=["월전세거래량"],
)
gu_market_features = add_lag_features(
    gu_panel,
    group_cols=["sggCd"],
    prefix="자치구",
    market_name="전세",
    price_col="월중위전세평당가",
    volume_col="월전세거래량",
)

dong_monthly = (
    jeonse.groupby(["sggCd", "umdNm", "거래월일자"])
    .agg(
        월중위전세평당가=("전세평당가", "median"),
        월전세거래량=("전세평당가", "size"),
    )
    .reset_index()
)
dong_panel = complete_month_panel(
    dong_monthly,
    group_cols=["sggCd", "umdNm"],
    count_cols=["월전세거래량"],
)
dong_market_features = add_lag_features(
    dong_panel,
    group_cols=["sggCd", "umdNm"],
    prefix="법정동",
    market_name="전세",
    price_col="월중위전세평당가",
    volume_col="월전세거래량",
)

dong_feature_cols = [
    col for col in dong_market_features.columns
    if col.startswith("법정동_")
]

print("자치구×월:", gu_market_features.shape)
print("법정동×월:", dong_market_features.shape)
display(
    dong_market_features[
        ["sggCd", "umdNm", "거래월일자"] + dong_feature_cols
    ].tail(10)
)

자치구×월: (1650, 10)
법정동×월: (22242, 11)


,sggCd,umdNm,거래월일자,법정동_전월_전세평당가,법정동_전월_전세거래량,법정동_과거3개월_전세평당가,법정동_과거3개월_전세거래량,법정동_과거6개월_전세평당가,법정동_과거6개월_전세거래량
22232,11740,천호동,2025-09-01,2334.329763,47.0,2334.329763,176.0,2379.815105,356.0
22233,11740,천호동,2025-10-01,2338.444725,56.0,2338.444725,176.0,2381.872586,347.0
22234,11740,천호동,2025-11-01,2393.930217,77.0,2338.444725,180.0,2366.187471,356.0
22235,11740,천호동,2025-12-01,2587.703436,52.0,2393.930217,185.0,2366.187471,361.0
22236,11740,천호동,2026-01-01,2352.882562,59.0,2393.930217,188.0,2373.406390,364.0
22237,11740,천호동,2026-02-01,2567.98697,59.0,2567.986970,170.0,2373.406390,350.0
22238,11740,천호동,2026-03-01,2508.144309,40.0,2508.144309,158.0,2451.037263,343.0
22239,11740,천호동,2026-04-01,2663.510042,52.0,2567.986970,151.0,2538.065639,339.0
22240,11740,천호동,2026-05-01,3579.305637,43.0,2663.510042,135.0,2577.845203,305.0
22241,11740,천호동,2026-06-01,2543.656824,41.0,2663.510042,136.0,2555.821897,294.0


## 12. 단지 단위 피처와 fallback

`aptSeq`를 이용해 단지별 과거 임대시장 정보를 생성한다.

- `단지_전월_전세평당가`
- `단지_과거3개월_전세평당가`
- `단지_과거6개월_전세평당가`
- `단지_전월_전세거래량`

단지별 거래량이 부족하면 결측치가 많이 발생할 수 있으므로 다음과 같은 fallback 구조를 고려한다.

1. 단지 정보가 충분하면 단지 전세가격 사용
2. 단지 정보가 부족하면 법정동 전세가격 사용
3. 법정동 정보도 부족하면 자치구 전세가격 사용

---

In [ ]:
apt_source = jeonse.loc[jeonse["aptSeq"].notna()].copy()
apt_meta = (
    apt_source.sort_values("거래일자")
    .drop_duplicates("aptSeq", keep="last")
    [["aptSeq", "aptNm", "sggCd", "umdNm"]]
)

apt_monthly = (
    apt_source.groupby(["aptSeq", "sggCd", "umdNm", "거래월일자"])
    .agg(
        월중위전세평당가=("전세평당가", "median"),
        월전세거래량=("전세평당가", "size"),
    )
    .reset_index()
)
apt_panel = complete_month_panel(
    apt_monthly,
    group_cols=["aptSeq", "sggCd", "umdNm"],
    count_cols=["월전세거래량"],
)
apt_market_features = add_lag_features(
    apt_panel,
    group_cols=["aptSeq", "sggCd", "umdNm"],
    prefix="단지",
    market_name="전세",
    price_col="월중위전세평당가",
    volume_col="월전세거래량",
)
apt_market_features = apt_market_features.merge(
    apt_meta[["aptSeq", "aptNm"]],
    on="aptSeq",
    how="left",
    validate="many_to_one",
)

dong_cols = [
    col for col in dong_market_features.columns
    if col.startswith("법정동_")
]
gu_cols = [
    col for col in gu_market_features.columns
    if col.startswith("자치구_")
]

apt_market_features = apt_market_features.merge(
    dong_market_features[
        ["sggCd", "umdNm", "거래월일자"] + dong_cols
    ],
    on=["sggCd", "umdNm", "거래월일자"],
    how="left",
    validate="many_to_one",
)
apt_market_features = apt_market_features.merge(
    gu_market_features[["sggCd", "거래월일자"] + gu_cols],
    on=["sggCd", "거래월일자"],
    how="left",
    validate="many_to_one",
)

for label, min_trades in [
    ("전월", 1),
    ("과거3개월", APT_MIN_TRADES),
    ("과거6개월", APT_MIN_TRADES),
]:
    apt_volume = apt_market_features[f"단지_{label}_전세거래량"]
    apt_price = apt_market_features[f"단지_{label}_전세평당가"].where(
        apt_volume >= min_trades
    )
    dong_price = apt_market_features[f"법정동_{label}_전세평당가"]
    gu_price = apt_market_features[f"자치구_{label}_전세평당가"]

    
    # 가격만 fallback
    apt_market_features[f"전세평당가_{label}"] = (
        apt_price
        .combine_first(dong_price)
        .combine_first(gu_price)
    )

    # 가격이 어느 공간 단위에서 왔는지 기록
    apt_market_features[f"전세피처출처_{label}"] = np.select(
        [
            apt_price.notna(),
            apt_price.isna() & dong_price.notna(),
            apt_price.isna() & dong_price.isna() & gu_price.notna(),
        ],
        ["단지", "법정동", "자치구"],
        default="결측",
    )

print("단지 수:", apt_market_features["aptSeq"].nunique())
display(
    apt_market_features[
        [
            "전세피처출처_전월",
            "전세피처출처_과거3개월",
            "전세피처출처_과거6개월",
        ]
    ].apply(pd.Series.value_counts).fillna(0)
)

단지 수: 8691


,전세피처출처_전월,전세피처출처_과거3개월,전세피처출처_과거6개월
법정동,367525,399725,364270
단지,190200,149923,178051
결측,8706,17412,26118
자치구,8165,7536,6157


## 13. 전세가율 피처

과거 전세시장 가격과 과거 매매시장 가격을 결합해 전세가율을 생성한다.

현재 전세가격을 현재 개별 거래금액으로 나누면 타깃 정보가 섞일 수 있으므로 사용하지 않는다. 올바른 방식은 동일한 과거 기간의 시장 집계끼리 비교하는 것이다.

예시:

> 법정동 전월 전세평당가 ÷ 법정동 전월 매매평당가

이를 통해 해당 지역에서 매매가격 대비 전세가격이 어느 정도 수준인지 표현한다.

---

In [ ]:
trade_features = pd.read_parquet(TRADE_FEATURE_PATH)
required_trade_cols = [
    "법정동시군구코드",
    "법정동",
    "단지일련번호",
    "거래월일자",
    "평당가",
]
missing_trade_cols = [
    col for col in required_trade_cols if col not in trade_features.columns
]
if missing_trade_cols:
    raise KeyError(f"매매 피처 파일에 필요한 컬럼이 없습니다: {missing_trade_cols}")

trade_features["법정동시군구코드"] = (
    trade_features["법정동시군구코드"]
    .astype("string")
    .str.split(".")
    .str[0]
    .str.zfill(5)
)
trade_features["법정동"] = trade_features["법정동"].astype("string").str.strip()
trade_features["단지일련번호"] = (
    trade_features["단지일련번호"].astype("string").str.strip()
)
trade_features["거래월일자"] = pd.to_datetime(trade_features["거래월일자"])

trade_gu_monthly = (
    trade_features.groupby(["법정동시군구코드", "거래월일자"])
    .agg(
        월중위매매평당가=("평당가", "median"),
        월매매거래량=("평당가", "size"),
    )
    .reset_index()
    .rename(columns={"법정동시군구코드": "sggCd"})
)
trade_gu_panel = complete_month_panel(
    trade_gu_monthly,
    group_cols=["sggCd"],
    count_cols=["월매매거래량"],
)
trade_gu_features = add_lag_features(
    trade_gu_panel,
    group_cols=["sggCd"],
    prefix="자치구",
    market_name="매매",
    price_col="월중위매매평당가",
    volume_col="월매매거래량",
)

trade_dong_monthly = (
    trade_features.groupby(
        ["법정동시군구코드", "법정동", "거래월일자"]
    )
    .agg(
        월중위매매평당가=("평당가", "median"),
        월매매거래량=("평당가", "size"),
    )
    .reset_index()
    .rename(
        columns={
            "법정동시군구코드": "sggCd",
            "법정동": "umdNm",
        }
    )
)
trade_dong_panel = complete_month_panel(
    trade_dong_monthly,
    group_cols=["sggCd", "umdNm"],
    count_cols=["월매매거래량"],
)
trade_dong_features = add_lag_features(
    trade_dong_panel,
    group_cols=["sggCd", "umdNm"],
    prefix="법정동",
    market_name="매매",
    price_col="월중위매매평당가",
    volume_col="월매매거래량",
)

trade_gu_cols = [
    col for col in trade_gu_features.columns if col.startswith("자치구_")
]
trade_dong_cols = [
    col for col in trade_dong_features.columns if col.startswith("법정동_")
]
gu_market_features = gu_market_features.merge(
    trade_gu_features[["sggCd", "거래월일자"] + trade_gu_cols],
    on=["sggCd", "거래월일자"],
    how="left",
    validate="one_to_one",
)
dong_market_features = dong_market_features.merge(
    trade_dong_features[
        ["sggCd", "umdNm", "거래월일자"] + trade_dong_cols
    ],
    on=["sggCd", "umdNm", "거래월일자"],
    how="left",
    validate="one_to_one",
)

for label in ["전월", "과거3개월", "과거6개월"]:
    dong_sale_price = dong_market_features[
        f"법정동_{label}_매매평당가"
    ].replace(0, np.nan)
    gu_sale_price = gu_market_features[
        f"자치구_{label}_매매평당가"
    ].replace(0, np.nan)

    dong_market_features[f"법정동_{label}_전세가율"] = (
        dong_market_features[f"법정동_{label}_전세평당가"]
        / dong_sale_price
    )
    gu_market_features[f"자치구_{label}_전세가율"] = (
        gu_market_features[f"자치구_{label}_전세평당가"]
        / gu_sale_price
    )

ratio_cols = [
    "법정동_전월_전세가율",
    "법정동_과거3개월_전세가율",
    "법정동_과거6개월_전세가율",
]
display(dong_market_features[ratio_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
법정동_전월_전세가율,13404.0,0.555914,0.178642,0.10138,0.449409,0.535608,0.627141,2.968747
법정동_과거3개월_전세가율,13401.0,0.550001,0.154013,0.135814,0.45782,0.536191,0.616219,2.069109
법정동_과거6개월_전세가율,13780.0,0.551718,0.14802,0.141845,0.463706,0.539299,0.614109,2.102399


## 14. 매매 데이터와 결합

전월세 피처 생성이 완료되면 기존 아파트 매매 피처 데이터와 거래월·지역·단지 키를 기준으로 결합한다.

최종 매매 거래 행에는 다음 정보가 포함된다.

### 주택 특성

- 전용면적
- 층
- 건축연도
- 연식

### 입지 특성

- 자치구·법정동
- 지하철 거리
- 학교 거리
- 기타 생활 인프라

### 매매시장

- 전월 매매가격
- 과거 3개월·6개월 매매가격
- 거래량
- 가격 변화율

### 임대시장

- 전월 전세가격
- 과거 3개월·6개월 전세가격
- 전세 거래량
- 전세가율

### 금융시장

- 주택담보대출 금리
- 전세자금대출 금리
- 한국은행 기준금리
- 서울 주택담보대출 월액 등

---

In [42]:
trade_rent_features = trade_features.copy()
before_merge_rows = len(trade_rent_features)

gu_feature_cols = [
    col for col in gu_market_features.columns if col.startswith("자치구_")
]
dong_feature_cols = [
    col for col in dong_market_features.columns if col.startswith("법정동_")
]
apt_feature_cols = [
    col for col in apt_market_features.columns
    if col.startswith("단지_")
    or col.startswith("전세평당가_")
    or col.startswith("전세거래량_")
    or col.startswith("전세피처출처_")
]

gu_for_merge = gu_market_features[
    ["sggCd", "거래월일자"] + gu_feature_cols
].rename(columns={"sggCd": "법정동시군구코드"})
dong_for_merge = dong_market_features[
    ["sggCd", "umdNm", "거래월일자"] + dong_feature_cols
].rename(
    columns={
        "sggCd": "법정동시군구코드",
        "umdNm": "법정동",
    }
)
apt_for_merge = apt_market_features[
    ["aptSeq", "sggCd", "umdNm", "거래월일자"] + apt_feature_cols
].rename(
    columns={
        "aptSeq": "단지일련번호",
        "sggCd": "법정동시군구코드",
        "umdNm": "법정동",
    }
)

trade_rent_features = trade_rent_features.merge(
    gu_for_merge,
    on=["법정동시군구코드", "거래월일자"],
    how="left",
    validate="many_to_one",
)
trade_rent_features = trade_rent_features.merge(
    dong_for_merge,
    on=["법정동시군구코드", "법정동", "거래월일자"],
    how="left",
    validate="many_to_one",
)
trade_rent_features = trade_rent_features.merge(
    apt_for_merge,
    on=["단지일련번호", "법정동시군구코드", "법정동", "거래월일자"],
    how="left",
    validate="many_to_one",
)

for label, min_trades in [
    ("전월", 1),
    ("과거3개월", APT_MIN_TRADES),
    ("과거6개월", APT_MIN_TRADES),
]:
    apt_volume = trade_rent_features[f"단지_{label}_전세거래량"]
    apt_price = trade_rent_features[f"단지_{label}_전세평당가"].where(
        apt_volume >= min_trades
    )
    dong_price = trade_rent_features[f"법정동_{label}_전세평당가"]
    gu_price = trade_rent_features[f"자치구_{label}_전세평당가"]

    trade_rent_features[f"전세평당가_{label}"] = (
        apt_price.combine_first(dong_price).combine_first(gu_price)
    )
    trade_rent_features[f"전세거래량_{label}"] = (
        apt_volume.where(apt_price.notna())
        .combine_first(trade_rent_features[f"법정동_{label}_전세거래량"])
        .combine_first(trade_rent_features[f"자치구_{label}_전세거래량"])
    )
    trade_rent_features[f"전세피처출처_{label}"] = np.select(
        [
            apt_price.notna(),
            apt_price.isna() & dong_price.notna(),
            apt_price.isna() & dong_price.isna() & gu_price.notna(),
        ],
        ["단지", "법정동", "자치구"],
        default="결측",
    )

if len(trade_rent_features) != before_merge_rows:
    raise AssertionError("전월세 피처 결합 중 매매 행 수가 변경되었습니다.")

rent_feature_cols = [
    col for col in trade_rent_features.columns
    if col.startswith("자치구_")
    or col.startswith("법정동_")
    or col.startswith("단지_")
    or col.startswith("전세평당가_")
    or col.startswith("전세거래량_")
    or col.startswith("전세피처출처_")
]

print("결합 전/후 행 수:", before_merge_rows, "→", len(trade_rent_features))
print("추가 전월세 피처 수:", len(rent_feature_cols))
display(
    trade_rent_features[rent_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("결측비율")
)

결합 전/후 행 수: 234302 → 234302
추가 전월세 피처 수: 45


,결측비율
단지_전월_전세평당가,0.183575
단지_과거3개월_전세평당가,0.168223
단지_과거6개월_전세평당가,0.139026
법정동_과거6개월_전세가율,0.056436
법정동_과거6개월_매매평당가,0.055877
자치구_과거6개월_전세가율,0.048770
자치구_과거6개월_매매평당가,0.048770
법정동_과거3개월_전세가율,0.048327
법정동_과거3개월_매매평당가,0.047362
자치구_과거3개월_매매평당가,0.037289


## 15. 시계열 누수 방지

매매가격을 예측하는 거래월을 `t`라고 하면 시장 관련 피처는 `t-1`과 그 이전 정보만 사용한다.

예를 들어 2025년 6월 거래를 예측할 때 사용할 수 있는 정보는 다음과 같다.

- 2025년 5월 전세시장
- 2025년 3~5월 전세시장
- 2024년 12월~2025년 5월 전세시장

사용하면 안 되는 정보는 다음과 같다.

- 2025년 6월 전체 전세시장 집계
- 2025년 6월 전체 매매시장 집계
- 현재 거래금액으로부터 계산한 피처

---

In [43]:
def values_match(actual, expected):
    if pd.isna(actual) and pd.isna(expected):
        return True
    if pd.isna(actual) or pd.isna(expected):
        return False
    return np.isclose(float(actual), float(expected), rtol=1e-10, atol=1e-10)


audit_candidates = dong_market_features.dropna(
    subset=["법정동_과거6개월_전세평당가"]
).head(20)
audit_records = []

for _, row in audit_candidates.iterrows():
    target_month = row["거래월일자"]
    group_mask = (
        dong_panel["sggCd"].eq(row["sggCd"])
        & dong_panel["umdNm"].eq(row["umdNm"])
    )

    for window in ROLLING_WINDOWS:
        start_month = target_month - pd.DateOffset(months=window)
        source_rows = dong_panel.loc[
            group_mask
            & dong_panel["거래월일자"].ge(start_month)
            & dong_panel["거래월일자"].lt(target_month)
        ]
        valid_prices = source_rows["월중위전세평당가"].dropna()
        expected_price = (
            valid_prices.median()
            if len(valid_prices) >= PRICE_MIN_MONTHS[window]
            else np.nan
        )
        expected_volume = (
            source_rows["월전세거래량"].sum()
            if len(source_rows) == window
            else np.nan
        )
        actual_price = row[f"법정동_과거{window}개월_전세평당가"]
        actual_volume = row[f"법정동_과거{window}개월_전세거래량"]

        price_ok = values_match(actual_price, expected_price)
        volume_ok = values_match(actual_volume, expected_volume)
        source_before_target = (
            source_rows.empty
            or source_rows["거래월일자"].max() < target_month
        )

        audit_records.append(
            {
                "sggCd": row["sggCd"],
                "umdNm": row["umdNm"],
                "대상월": target_month,
                "window": window,
                "가격일치": price_ok,
                "거래량일치": volume_ok,
                "원천월_대상월이전": source_before_target,
            }
        )

lag_audit = pd.DataFrame(audit_records)
if not lag_audit.empty:
    assert lag_audit[
        ["가격일치", "거래량일치", "원천월_대상월이전"]
    ].all().all()

target_derived_blocklist = {
    "거래금액",
    "log_price",
    "평당가",
    "log_평당가",
}
if target_derived_blocklist.intersection(rent_feature_cols):
    raise AssertionError("현재 거래가격 기반 컬럼이 전월세 피처에 포함되었습니다.")

print("lag 표본 검증:", len(lag_audit), "건")
print("누수 금지 컬럼과 전월세 피처 교집합: 없음")
display(lag_audit.head(20))

lag 표본 검증: 40 건
누수 금지 컬럼과 전월세 피처 교집합: 없음


,sggCd,umdNm,대상월,window,가격일치,거래량일치,원천월_대상월이전
0,11110,교북동,2021-06-01,3,True,True,True
1,11110,교북동,2021-06-01,6,True,True,True
2,11110,교북동,2021-07-01,3,True,True,True
3,11110,교북동,2021-07-01,6,True,True,True
4,11110,교북동,2021-08-01,3,True,True,True
5,11110,교북동,2021-08-01,6,True,True,True
6,11110,교북동,2021-09-01,3,True,True,True
7,11110,교북동,2021-09-01,6,True,True,True
8,11110,교북동,2021-10-01,3,True,True,True
9,11110,교북동,2021-10-01,6,True,True,True


## 16. 최종 전월세 처리 흐름

국토교통부 전월세 API 수집  
→ 서울 25개 구 × 66개월 원천 데이터 확보  
→ Raw 데이터 통합  
→ 결측치·중복·이상치 점검  
→ 숫자형 및 날짜 전처리  
→ 전세·월세 구분  
→ 전세 평당가 생성  
→ 전월세 EDA  
→ 법정동 월별 전세시장 집계  
→ 단지 월별 전세시장 집계  
→ `shift(1)`을 이용한 과거 피처 생성  
→ 3개월·6개월 rolling 피처 생성  
→ 과거 매매시장과 결합  
→ 전세가율 생성  
→ 금융 데이터 결합  
→ 최종 매매 Feature Dataset 생성  
→ Clustering  
→ Regression  
→ Classification

In [44]:
output_manifest = pd.DataFrame(
    [
        {
            "데이터": "자치구 월별 전월세 피처",
            "경로": str(RENT_GU_PATH),
            "행": len(gu_market_features),
            "열": len(gu_market_features.columns),
        },
        {
            "데이터": "법정동 월별 전월세 피처",
            "경로": str(RENT_DONG_PATH),
            "행": len(dong_market_features),
            "열": len(dong_market_features.columns),
        },
        {
            "데이터": "단지 월별 전월세 피처",
            "경로": str(RENT_APT_PATH),
            "행": len(apt_market_features),
            "열": len(apt_market_features.columns),
        },
        {
            "데이터": "매매+전월세 최종 피처",
            "경로": str(TRADE_RENT_FEATURE_PATH),
            "행": len(trade_rent_features),
            "열": len(trade_rent_features.columns),
        },
    ]
)

if SAVE_OUTPUTS:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    gu_market_features.to_parquet(RENT_GU_PATH, index=False)
    dong_market_features.to_parquet(RENT_DONG_PATH, index=False)
    apt_market_features.to_parquet(RENT_APT_PATH, index=False)
    trade_rent_features.to_parquet(TRADE_RENT_FEATURE_PATH, index=False)
    print("저장 완료")
else:
    print("SAVE_OUTPUTS=False: 검증만 수행하고 파일은 저장하지 않았습니다.")

display(output_manifest)

SAVE_OUTPUTS=False: 검증만 수행하고 파일은 저장하지 않았습니다.


,데이터,경로,행,열
0,자치구 월별 전월세 피처,c:\Users\uk\Apartment price\data\v2_5year_seou...,1650,19
1,법정동 월별 전월세 피처,c:\Users\uk\Apartment price\data\v2_5year_seou...,22242,20
2,단지 월별 전월세 피처,c:\Users\uk\Apartment price\data\v2_5year_seou...,574596,34
3,매매+전월세 최종 피처,c:\Users\uk\Apartment price\data\v2_5year_seou...,234302,125
